In [1]:
!pip install polars

In [2]:
"""
Script de ETL e Harmonização - Base BPS
Processamento otimizado utilizando a Lazy API do Polars.
"""

import pandas as pd
import re
import logging
import polars as pl
from pathlib import Path

In [3]:
# ── CAMINHO DOS DADOS ────────────────────────────────────────────────────
# Ajuste PASTA_DADOS para o local onde estao os arquivos.
# Padrao: subpasta "dados" ao lado do notebook. No Google Colab, aponte para
# a pasta do seu Drive apos monta-lo.
import os
PASTA_DADOS = os.environ.get('BPS_DADOS', 'dados')
# ─────────────────────────────────────────────────────────────────────────


Mounted at /content/drive


In [4]:
# ── 1. CONFIGURAÇÕES E DIRETÓRIOS ────────────────────────────────────────────
# ⚙️ AJUSTE OS CAMINHOS ABAIXO SE NECESSÁRIO
PASTA_CONVERSAO = Path(PASTA_DADOS) / 'Base Conversao CSV'
PASTA_TRATADOS  = Path(PASTA_DADOS) / 'Base Harmonizacao Campos'

Path(PASTA_TRATADOS).mkdir(parents=True, exist_ok=True)

LOG_FILE = Path(PASTA_TRATADOS) / 'etl_execucao.log'

# Reseta handlers para evitar duplicação de log ao reexecutar a célula
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE, encoding='utf-8', mode='a'),
        logging.StreamHandler()
    ]
)

csvs = sorted(Path(PASTA_CONVERSAO).glob('BPS_*.csv'))
logging.info(f'Configuração OK | CSVs encontrados: {len(csvs)} | Log: {LOG_FILE}')
for f in csvs:
    print(f'  • {f.name}')

2026-07-02 01:26:03,874 - INFO - Configuração OK | CSVs encontrados: 26 | Log: /content/drive/MyDrive/TCC/Registro de Compras v2/Base v2/Base Harmonizacao Campos/etl_execucao.log


  • BPS_2000.csv
  • BPS_2001.csv
  • BPS_2002.csv
  • BPS_2003.csv
  • BPS_2004.csv
  • BPS_2005.csv
  • BPS_2006.csv
  • BPS_2007.csv
  • BPS_2008.csv
  • BPS_2009.csv
  • BPS_2010.csv
  • BPS_2011.csv
  • BPS_2012.csv
  • BPS_2013.csv
  • BPS_2014.csv
  • BPS_2015.csv
  • BPS_2016.csv
  • BPS_2017.csv
  • BPS_2018.csv
  • BPS_2019.csv
  • BPS_2020.csv
  • BPS_2021.csv
  • BPS_2022.csv
  • BPS_2023.csv
  • BPS_2024.csv
  • BPS_2025.csv


In [5]:
# ── 2. DICIONÁRIOS DE HARMONIZAÇÃO ───────────────────────────────────────────
RENOMEAR = {
    '\xa0Código BR\xa0': 'cod_catmat', 'Código BR': 'cod_catmat', 'Código CATMAT': 'cod_catmat',
    'Descrição do item': 'desc_item', '\xa0Descrição Item\xa0': 'desc_item', 'Descrição\xa0Item': 'desc_item',
    'Descrição\xa0CATMAT': 'desc_item', 'Descrição': 'desc_item', 'Descrição Catmat': 'desc_item',
    'Descrição CATMAT': 'desc_item',   # 2023-2024
    'ANVISA': 'registro_anvisa',         # 2023-2024

    'Unidade de fornecimento': 'unidade_fornecimento', '\xa0Unidade de Fornecimento\xa0': 'unidade_fornecimento',
    '\xa0Unidade de \nFornecimento\xa0': 'unidade_fornecimento', 'Unidade de\nfornecimento': 'unidade_fornecimento',
    'Unidade Fornecimento': 'unidade_fornecimento',
    'Preço unitário': 'preco_unitario', '\xa0Preço Unitário\xa0': 'preco_unitario', 'Preço Unitário': 'preco_unitario',
    'Unitário': 'preco_unitario',
    'Pago': 'preco_unitario',              # 2015
 'Valor Item Compra': 'preco_unitario',
    'Quantidade comprada': 'quantidade', '\xa0Qtd Itens Comprados\xa0': 'quantidade', 'Qtd Itens Comprados': 'quantidade',
    'Quantidade': 'quantidade', 'Quantidade Item Compra': 'quantidade',
    'Valor total do item': 'valor_total', 'Total': 'valor_total', 'Preço Total': 'valor_total', 'Valor Total Compra': 'valor_total',
    'Data da compra': 'data_compra', '\xa0Data Compra\xa0': 'data_compra', 'Compra': 'data_compra', 'Data Homologação': 'data_compra',
    'Modalidade de compra': 'modalidade_compra', '\xa0Modalidade de Compra\xa0': 'modalidade_compra', 'Modalidade da Compra': 'modalidade_compra',
    'Modalidade \nda compra': 'modalidade_compra', 'Modalidade\nda compra': 'modalidade_compra', 'Modalidade Compra': 'modalidade_compra',
    '\xa0Tipo Compra\xa0': 'tipo_compra', 'Tipo Compra': 'tipo_compra', 'Tipo': 'tipo_compra',
    'UF': 'uf', '\xa0UF\xa0': 'uf',
    '\xa0Município\xa0': 'municipio', 'Município Instituição': 'municipio', 'Nome Município': 'municipio',
    'Fornecedor': 'fornecedor', '\xa0Fornecedor\xa0': 'fornecedor', 'Nome Fornecedor': 'fornecedor',
    'Nome do fornecedor': 'fornecedor',    # 2020

    '\xa0CNPJ Fornecedor\xa0': 'cnpj_fornecedor', 'CNPJ Fornecedor': 'cnpj_fornecedor',
    '\xa0Fabricante\xa0': 'fabricante', 'Fabricante': 'fabricante', 'Nome Fabricante': 'fabricante',
    'Nome do fabricante': 'fabricante',    # 2020

    '\xa0CNPJ Fabricante\xa0': 'cnpj_fabricante', 'CNPJ Fabricante': 'cnpj_fabricante',
    'Instituição compradora': 'instituicao', '\xa0Nome Instituição\xa0': 'instituicao', 'Nome Instituição': 'instituicao',
    'Instituição': 'instituicao', 'Nome da Instituição': 'instituicao',
    'Nome da instituição': 'instituicao',  # 2020 (minúsculo)

    '\xa0CNPJ Instituição\xa0': 'cnpj_instituicao', 'CNPJ Instituição': 'cnpj_instituicao', 'CNPJ Comprador': 'cnpj_instituicao',
    '\xa0Registro Anvisa\xa0': 'registro_anvisa', 'Registro Anvisa': 'registro_anvisa', 'Anvisa': 'registro_anvisa',
    '\xa0Genérico\xa0': 'generico', 'Genérico': 'generico',
    'Classe': 'classe_catmat',
    '\xa0Licitação\xa0': 'licitacao', 'Licitação': 'licitacao',
    '\xa0Nota Fiscal\xa0': 'nota_fiscal', 'Nota Fiscal': 'nota_fiscal',
    '\xa0Esfera\xa0': 'esfera', 'Esfera': 'esfera',
    '\xa0Média Ponderada\xa0': 'media_ponderada', 'Inserção': 'data_insercao', '\xa0Data Inserção\xa0': 'data_insercao', '\xa0Data Inserção': 'data_insercao',
    'CMED - Preço Regulado': 'cmed_preco_regulado', 'Competência CMED': 'cmed_competencia', 'Região': 'regiao', 'País': 'pais',
    'Qualificação': 'qualificacao_fornecedor', 'Pago': 'preco_unitario',              # 2015, '\xa0Maior Preço': 'maior_preco', '\xa0Menor Preço': 'menor_preco',
    'Código Compra': 'codigo_compra', 'Seq. Compra Item': 'seq_compra_item', 'Observações': 'observacoes'
}

COLUNAS_FINAIS = [
    'ano', 'data_compra', 'cod_catmat', 'desc_item', 'unidade_fornecimento', 'preco_unitario', 'quantidade',
    'valor_total', 'modalidade_compra', 'tipo_compra', 'uf', 'municipio', 'regiao', 'instituicao', 'cnpj_instituicao',
    'fornecedor', 'cnpj_fornecedor', 'fabricante', 'cnpj_fabricante', 'registro_anvisa', 'generico', 'classe_catmat', 'esfera'
]

**Função principal de ETL — versão vigente (v2).** Inclui a correção do separador decimal (regra formato-ciente) e a correção da inversão dos rótulos de CNPJ/Fornecedor (2018–2020). A versão anterior (v1) está preservada no arquivo Historico_Notebooks_TCC.md.

In [6]:
# ── 3. FUNÇÃO PRINCIPAL DE ETL (LAZY API COM PROTEÇÃO E INVERSÃO DE CNPJ) ────
def harmonizar_arquivo_polars(csv_origem: Path, pasta_saida: Path):
    ano      = int(re.search(r'(\d{4})', csv_origem.name).group(1))
    parq_out = pasta_saida / f'BPS_{ano}.parquet'

    if parq_out.exists():
        parq_out.unlink()

    logging.info(f'Processando: {csv_origem.name}')

    try:
        # ── AJUSTE: Dicionário dinâmico para tratar a inversão de 2018 a 2020
        dicionario_atual = RENOMEAR.copy()
        if ano in [2018, 2019, 2020]:
            dicionario_atual.update({
                'Fornecedor': 'cnpj_fornecedor',
                'Nome Fornecedor': 'fornecedor',
                'Nome do fornecedor': 'fornecedor',
                'Fabricante': 'cnpj_fabricante',
                'Nome Fabricante': 'fabricante',
                'Nome do fabricante': 'fabricante',
                'Instituição': 'cnpj_instituicao',
                'Nome da Instituição': 'instituicao',
                'Nome da instituição': 'instituicao'
            })

        # ── Lê só o cabeçalho com Pandas
        df_cab = pd.read_csv(
            csv_origem, nrows=0, encoding='utf-8-sig',
            dtype=str, on_bad_lines='skip'
        )
        colunas_originais = df_cab.columns.tolist()
        logging.info(f'  Colunas originais ({len(colunas_originais)}): {colunas_originais}')

        # ── Filtra redundâncias usando o dicionário específico do ano
        nomes_canonicos_vistos = set()
        colunas_seguras        = []
        colunas_descartadas    = []

        for col in colunas_originais:
            col_limpo   = col.replace('\xa0', ' ').replace('\n', ' ').strip()
            nome_canon  = dicionario_atual.get(col_limpo, dicionario_atual.get(col, col_limpo))

            if nome_canon in nomes_canonicos_vistos and nome_canon != col_limpo:
                colunas_seguras.append(f'__descartada_{col_limpo}__')
                colunas_descartadas.append(col)
            else:
                colunas_seguras.append(col)
                nomes_canonicos_vistos.add(nome_canon)

        if colunas_descartadas:
            logging.info(f'  Colunas redundantes descartadas: {colunas_descartadas}')

        # ── Lê CSV com Polars
        lazy_df = pl.scan_csv(
            csv_origem,
            infer_schema_length=0,
            ignore_errors=True,
            encoding="utf8-lossy",
            null_values=["", "NA", "null", "NaN"],
            has_header=False,
            skip_rows=1,
            new_columns=colunas_seguras
        )

        # ── Descarta colunas redundantes
        cols_para_dropar = [c for c in colunas_seguras if c.startswith('__descartada_')]
        if cols_para_dropar:
            lazy_df = lazy_df.drop(cols_para_dropar)

        # ── Renomeia usando o dicionário atualizado
        colunas_atuais = lazy_df.collect_schema().names()
        mapa_renomear  = {}
        for col in colunas_atuais:
            col_limpo = col.replace('\xa0', ' ').replace('\n', ' ').strip()
            if col_limpo in dicionario_atual:
                mapa_renomear[col] = dicionario_atual[col_limpo]
            elif col in dicionario_atual:
                mapa_renomear[col] = dicionario_atual[col]
            elif col_limpo in dicionario_atual.values():
                mapa_renomear[col] = col_limpo

        lazy_df = lazy_df.rename(mapa_renomear)

        # ── Seleciona só colunas canônicas presentes
        cols_presentes = [c for c in COLUNAS_FINAIS
                          if c in lazy_df.collect_schema().names()]
        lazy_df = lazy_df.select(cols_presentes)
        colunas_finais_atuais = lazy_df.collect_schema().names()
        logging.info(f'  Colunas canônicas selecionadas: {colunas_finais_atuais}')

        # ── Expressões e Tratamentos Finais
        exprs = []
        exprs.append(pl.lit(ano).cast(pl.Int32).alias('ano'))

        if 'data_compra' in colunas_finais_atuais:
            # ── CONVERSÃO DE DATA FORMATO-CIENTE (corrige 2º bug do \xa0)
            # Diagnóstico (set/2026): em 2009-2012 a data vem embrulhada no
            # espaço invisível, como '\xa001/01/2009\xa0'. Sem limpar o \xa0 e
            # aparar as pontas, TODOS os formatos de to_date falham e, com
            # strict=False, o resultado é null silencioso -> data_compra ficou
            # 100% nula nesses 4 anos (51.049 registros), inviabilizando a
            # deflacao mensal pelo mes da compra. Mesma familia do bug do
            # separador decimal, ja corrigido nas colunas numericas.
            base_dt = (pl.col('data_compra')
                       .str.replace_all("\u00a0", "", literal=True)
                       .str.strip_chars())
            exprs.append(
                pl.coalesce([
                    base_dt.str.to_date("%d/%m/%Y", strict=False),
                    base_dt.str.to_date("%Y-%m-%d", strict=False),
                    base_dt.str.to_date("%d/%m/%Y %H:%M:%S", strict=False),
                    base_dt.str.to_date("%Y-%m-%d %H:%M:%S", strict=False),
                ]).alias('data_compra')
            )

        for cnpj_col in ['cnpj_fornecedor', 'cnpj_fabricante', 'cnpj_instituicao']:
            if cnpj_col in colunas_finais_atuais:
                exprs.append(
                    pl.col(cnpj_col).fill_null("")
                    .str.replace_all(r"[^\d]", "")
                    .str.pad_start(14, '0')
                    .str.replace("00000000000000", "")
                    .alias(cnpj_col)
                )

        if 'uf' in colunas_finais_atuais:
            exprs.append(
                pl.col('uf').fill_null("").str.strip_chars()
                .str.to_uppercase().str.slice(0, 2).alias('uf')
            )

        if 'modalidade_compra' in colunas_finais_atuais:
            exprs.append(
                pl.col('modalidade_compra').fill_null("")
                .str.strip_chars().str.to_uppercase().alias('modalidade_compra')
            )

        # ── CONVERSÃO NUMÉRICA FORMATO-CIENTE (corrige bug do separador decimal)
        # Diagnóstico (jul/2026): nos anos em que o Excel guarda o preço como
        # célula NUMÉRICA (2000-2008 e 2014-2025), a conversão p/ CSV grava com
        # PONTO decimal ("1.95"). A regra antiga removia todo ponto como se
        # fosse milhar -> 1.95 virava 195 (corrupção de ~70-130x, confirmada
        # por comparação de medianas parquet vs. arquivos brutos).
        # Regra nova: só trata ponto como milhar QUANDO HÁ VÍRGULA na string
        # (formato BR, ex. "1.234,56" ou "0,0890" dos anos 2009-2013).
        # Sem vírgula, o ponto é decimal e converte direto.
        for num_col in ['preco_unitario', 'quantidade', 'valor_total']:
            if num_col in colunas_finais_atuais:
                base = (pl.col(num_col)
                        .str.replace_all("\u00a0", "", literal=True)  # remove \xa0
                        .str.strip_chars())                            # espaços das pontas
                exprs.append(
                    pl.when(base.str.contains(",", literal=True))
                      .then(base.str.replace_all(".", "", literal=True)   # milhar fora
                                .str.replace(",", ".", literal=True))     # vírgula -> decimal
                      .otherwise(base)                                    # ponto já é decimal
                      .cast(pl.Float64, strict=False)
                      .alias(num_col)
                )

        lazy_df = lazy_df.with_columns(exprs)

        # ════════════════════════════════════════════════════════════════════
        # 3.1 NORMALIZAÇÃO DE CONTEÚDO — cod_catmat e unidade_fornecimento
        # --------------------------------------------------------------------
        # NÃO altera QUAIS colunas existem (isso é a harmonização, acima).
        # Aqui padronizamos os VALORES, para que o mesmo item/embalagem seja
        # reconhecido como idêntico ao longo dos 17 anos. Regras validadas
        # contra os dados reais e a documentação oficial do BPS.
        # ════════════════════════════════════════════════════════════════════

        # ── 3.1.a  cod_catmat → "BR" + 7 dígitos (formato canônico oficial)
        #   • 7 díg sem "BR" (anos antigos)  → prefixa "BR"
        #   • 6 díg (2025, perdeu zero à esq.) → "BR0" + dígitos  (validado: reconecta com 2018)
        #   • "-", vazio, ou outro tamanho   → NULO (compra válida sem item rastreável;
        #     confirmado no Painel oficial do MS — fica fora das análises por item)
        if 'cod_catmat' in colunas_finais_atuais:
            lazy_df = lazy_df.with_columns(
                pl.col('cod_catmat').cast(pl.Utf8).str.replace_all(r"[^0-9]", "").alias('__catmat_d')
            )
            lazy_df = lazy_df.with_columns(
                pl.when(pl.col('__catmat_d').str.len_chars() == 0).then(None)
                  .when(pl.col('__catmat_d').str.len_chars() == 6).then("BR0" + pl.col('__catmat_d'))
                  .when(pl.col('__catmat_d').str.len_chars() == 7).then("BR"  + pl.col('__catmat_d'))
                  .otherwise(None).alias('cod_catmat')
            ).drop('__catmat_d')

        # ── 3.1.b  unidade_fornecimento → estrutura DUPLA (chave + rótulo)
        #   unidade_chave  : para AGRUPAR (entra na chave do item). Sem espaço,
        #                    sem acento, maiúsculas, decimal vírgula preservado
        #                    quando significativo (0,50→0,5; 100,00→100).
        #   unidade_rotulo : para EXIBIR no painel. Legível, acento preservado,
        #                    simples e consistente (refino estético fica p/ depois).
        if 'unidade_fornecimento' in colunas_finais_atuais:
            # limpeza comum (resolve \xa0, hífen, zeros decimais redundantes)
            _limpa = (
                pl.col('unidade_fornecimento').cast(pl.Utf8)
                  .str.replace_all("\u00a0", " ")                  # \xa0 → espaço
                  .str.replace_all("-", " ")                       # hífen (separador/nome) → espaço
                  .str.replace_all(r"(\d),00?(\D|$)", r"${1}${2}")  # remove ,00 e ,0 redundantes
                  .str.replace_all(r"(,\d*?)0+(\D|$)", r"${1}${2}") # remove zeros à direita (0,50→0,5)
                  .str.replace_all(r",(\s|$)", r"${1}")            # vírgula órfã
            )

            # unidade_chave: remove TODOS os espaços + maiúsculas + sem acento
            lazy_df = lazy_df.with_columns(
                _limpa.str.replace_all(r"\s+", "").str.to_uppercase().alias('unidade_chave')
            )
            lazy_df = lazy_df.with_columns(
                pl.col('unidade_chave')
                  .str.replace_all("Á","A").str.replace_all("À","A").str.replace_all("Â","A").str.replace_all("Ã","A")
                  .str.replace_all("É","E").str.replace_all("Ê","E").str.replace_all("Í","I")
                  .str.replace_all("Ó","O").str.replace_all("Ô","O").str.replace_all("Õ","O")
                  .str.replace_all("Ú","U").str.replace_all("Ç","C")
                  .alias('unidade_chave')
            )

            # unidade_rotulo: mantém espaço/acento, primeira letra maiúscula,
            # corrige siglas de medida que o titlecase rebaixa (Ml→ML etc.)
            lazy_df = lazy_df.with_columns(
                _limpa.str.replace_all(r"\s+", " ").str.strip_chars()
                      .str.to_titlecase().alias('unidade_rotulo')
            )
            lazy_df = lazy_df.with_columns(
                pl.col('unidade_rotulo')
                  .str.replace_all(r"\bMl\b","ML").str.replace_all(r"\bMg\b","MG")
                  .str.replace_all(r"\bMcg\b","MCG").str.replace_all(r"\bUn\b","UN")
                  .str.replace_all(r"(\d)Ml", r"${1}ML").str.replace_all(r"(\d)Mg", r"${1}MG")
                  .alias('unidade_rotulo')
            )
        # ════════════════════════════════════════════════════════════════════
        # FIM da normalização de conteúdo (3.1)
        # ════════════════════════════════════════════════════════════════════

        if ('valor_total' in colunas_finais_atuais
                and 'preco_unitario' in colunas_finais_atuais
                and 'quantidade' in colunas_finais_atuais):
            lazy_df = lazy_df.with_columns(
                pl.when(
                    pl.col('valor_total').is_null()
                    & pl.col('preco_unitario').is_not_null()
                    & pl.col('quantidade').is_not_null()
                )
                .then(pl.col('preco_unitario') * pl.col('quantidade'))
                .otherwise(pl.col('valor_total'))
                .alias('valor_total')
            )
        elif ('preco_unitario' in colunas_finais_atuais
              and 'quantidade' in colunas_finais_atuais):
            lazy_df = lazy_df.with_columns(
                (pl.col('preco_unitario') * pl.col('quantidade')).alias('valor_total')
            )

        filtro_cols = [c for c in ['preco_unitario', 'desc_item']
                       if c in colunas_finais_atuais]
        if filtro_cols:
            condicao = pl.all_horizontal([pl.col(c).is_null() for c in filtro_cols])
            lazy_df  = lazy_df.filter(~condicao)

        # ── Grava Parquet
        lazy_df.sink_parquet(parq_out)

        linhas_finais = pl.read_parquet(parq_out).select(pl.len()).item()
        logging.info(
            f'  Concluído: {parq_out.name} | {linhas_finais:,} linhas válidas ✓'
        )

    except Exception as e:
        logging.error(f'  ERRO em {csv_origem.name}: {type(e).__name__}: {e}')
        raise

print("Função harmonizar_arquivo_polars carregada ✓")

Função harmonizar_arquivo_polars carregada ✓


In [7]:
# ── 4. EXECUÇÃO EM LOTE ──────────────────────────────────────────────────────
logging.info('=' * 60)
logging.info('ETL Harmonização BPS — início')

resumo = []
for ano in range(2000, 2026):
    arquivo_csv = Path(PASTA_CONVERSAO) / f'BPS_{ano}.csv'
    if not arquivo_csv.exists():
        logging.warning(f'Arquivo não encontrado: BPS_{ano}.csv')
        resumo.append((ano, 'NÃO ENCONTRADO', 0))
        continue
    try:
        harmonizar_arquivo_polars(arquivo_csv, Path(PASTA_TRATADOS))
        parq = Path(PASTA_TRATADOS) / f'BPS_{ano}.parquet'
        linhas = pl.read_parquet(parq).select(pl.len()).item() if parq.exists() else 0
        resumo.append((ano, 'OK', linhas))
    except Exception as e:
        logging.error(f'FALHA BPS_{ano}: {e}')
        resumo.append((ano, f'ERRO: {type(e).__name__}', 0))

# Resumo final
logging.info('=' * 60)
logging.info('RESUMO FINAL')
total = 0
for ano, status, linhas in resumo:
    logging.info(f'  BPS_{ano}: {status} | {linhas:,} linhas')
    total += linhas
logging.info(f'Total de registros: {total:,}')
logging.info('ETL Harmonização BPS — concluído')

print(f'\nTotal de registros processados: {total:,}')

2026-07-02 01:26:46,790 - INFO - ============================================================
2026-07-02 01:26:46,791 - INFO - ETL Harmonização BPS — início
2026-07-02 01:26:46,797 - INFO - Processando: BPS_2000.csv
2026-07-02 01:26:47,458 - INFO -   Colunas originais (11): ['Descrição do item', 'Unidade de fornecimento', 'Instituição compradora', 'UF', 'Região', 'Data da compra', 'Modalidade de compra', 'Fornecedor', 'Quantidade comprada', 'Preço unitário', 'Valor total do item']
2026-07-02 01:26:47,568 - INFO -   Colunas canônicas selecionadas: ['data_compra', 'desc_item', 'unidade_fornecimento', 'preco_unitario', 'quantidade', 'valor_total', 'modalidade_compra', 'uf', 'regiao', 'instituicao', 'fornecedor']
2026-07-02 01:26:47,999 - INFO -   Concluído: BPS_2000.parquet | 7,330 linhas válidas ✓
2026-07-02 01:26:48,016 - INFO - Processando: BPS_2001.csv
2026-07-02 01:26:48,524 - INFO -   Colunas originais (11): ['Descrição do item', 'Unidade de fornecimento', 'Instituição compradora', 


Total de registros processados: 1,052,742


**Validação** — conferência de que `preco_unitario` e `valor_total` foram tratados corretamente.

In [8]:
import polars as pl
from pathlib import Path
PASTA = Path(Path(PASTA_DADOS) / 'Base Harmonizacao Campos')
for ano in [2009, 2010, 2011, 2012]:
    df = pl.read_parquet(PASTA / f'BPS_{ano}.parquet', columns=['preco_unitario','valor_total'])
    preco_ok = df.filter(pl.col('preco_unitario') > 0).height
    valor_ok = df.filter(pl.col('valor_total') > 0).height
    print(f'BPS_{ano}: preco>0: {preco_ok:,} | valor>0: {valor_ok:,}  (de {df.height:,})')

BPS_2009: preco>0: 13,494 | valor>0: 13,494  (de 13,494)
BPS_2010: preco>0: 19,067 | valor>0: 19,067  (de 19,069)
BPS_2011: preco>0: 13,611 | valor>0: 13,611  (de 13,611)
BPS_2012: preco>0: 4,875 | valor>0: 4,875  (de 4,875)
